# Milestone 0 — Data Preparation & Exploration

Category: **All_Beauty** (small, fast on free GPU).

Deliverables:
- EDA (rating, price, review length, image availability)
- Product-level train/val/test splits (no leakage)
- Cached image subset

In [ ]:
# Colab setup (uncomment on first run)
# !git clone https://github.com/YOUR_USER/smart-product-intelligence.git
# %cd smart-product-intelligence
# !pip install -q -r requirements.txt

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data import (
    DEFAULT_CATEGORY,
    MAX_IMAGES,
    MAX_PRODUCTS,
    MAX_REVIEWS,
    artifacts_dir,
    build_product_table,
    build_review_table,
    download_product_images,
    load_raw_datasets,
    merge_tables,
    save_splits,
    split_by_product,
)
from src.utils import setup_notebook_path

setup_notebook_path()
sns.set_theme(style="whitegrid")

In [ ]:
CATEGORY = DEFAULT_CATEGORY
print(f"Loading category: {CATEGORY}")
raw_reviews, raw_meta = load_raw_datasets(
    category=CATEGORY,
    max_products=MAX_PRODUCTS,
    max_reviews=MAX_REVIEWS,
)
print(raw_reviews.shape, raw_meta.shape)

In [ ]:
products = build_product_table(raw_meta)
reviews = build_review_table(raw_reviews)
products, reviews = merge_tables(products, reviews)
print(products.shape, reviews.shape)
products.head()

In [ ]:
splits = split_by_product(products, reviews)
save_splits(splits)
print({k: len(v) for k, v in splits.items()})

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
reviews["rating"].hist(bins=5, ax=axes[0, 0], color="#4C72B0")
axes[0, 0].set_title("Review rating distribution")

products["price"].dropna().hist(bins=30, ax=axes[0, 1], color="#55A868")
axes[0, 1].set_title("Price distribution")

reviews["review_length"].clip(0, 2000).hist(bins=40, ax=axes[1, 0], color="#C44E52")
axes[1, 0].set_title("Review length")

missing = products[["price", "average_rating", "image_url"]].isna().mean()
missing.plot(kind="bar", ax=axes[1, 1], color="#8172B2")
axes[1, 1].set_title("Missing data ratio (products)")
axes[1, 1].set_ylabel("fraction missing")
plt.tight_layout()
plt.savefig(artifacts_dir() / "eda_overview.png", dpi=150)
plt.show()

In [ ]:
image_manifest = download_product_images(splits["train_products"], max_images=MAX_IMAGES)
availability = image_manifest["image_available"].mean()
print(f"Image availability: {availability:.1%} ({image_manifest['image_available'].sum()} images)")

## Notes
- Splits are frozen under `data/splits/`.
- Run remaining notebooks only after this one completes.